## Baseline (1st Trial)
- This file serves as the baseline for my future trials
- The outcome is an RMSLE of 0.47
- This code simplifies the holidays dataset into is_holiday
- The added features are lag, is_holiday, calendar
- The categorical features are "family", "city", "state", 'type'

In [1]:
import pandas as pd
import numpy as np
import matplotlib

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Load Dataset

In [2]:
def load_dataset():
    holiday = pd.read_csv("holidays_events.csv", parse_dates=["date"])
    oil = pd.read_csv("oil.csv", parse_dates=["date"])
    train = pd.read_csv("train.csv", parse_dates=["date"])
    test = pd.read_csv("test.csv", parse_dates=["date"])
    stores = pd.read_csv("stores.csv")
    transactions = pd.read_csv("transactions.csv", parse_dates=["date"])
    return holiday, oil, train, test, stores, transactions

## Clean Oil

In [3]:
def preprocess_oil(oil: pd.DataFrame) -> pd.DataFrame:
    oil = oil.copy().sort_values("date")
    oil["dcoilwtico"] = oil["dcoilwtico"].ffill()
    return oil

## Holiday preprocessing

In [4]:
def preprocess_holiday_baseline(holiday: pd.DataFrame) -> pd.DataFrame:
    holiday = holiday.copy()
    holiday = holiday[holiday["transferred"] == False]

    # baseline: any real holiday/event/bridge/additional day counts as holiday,
    # but Work Day should not
    holiday["is_holiday"] = (holiday["type"] != "Work Day").astype(int)

    return holiday

## Merging dataset

In [5]:
def merge_base_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
    oil: pd.DataFrame,
    stores: pd.DataFrame,
    holiday_feat: pd.DataFrame,
):
    train = train.copy()
    test = test.copy()

    # oil
    train = train.merge(oil, on="date", how="left")
    test = test.merge(oil, on="date", how="left")

    # stores
    train = train.merge(stores, on="store_nbr", how="left")
    test = test.merge(stores, on="store_nbr", how="left")

    # holiday
    train = train.merge(holiday_feat, on="date", how="left")
    test = test.merge(holiday_feat, on="date", how="left")

    train["is_holiday"] = train["is_holiday"].fillna(0).astype(int)
    test["is_holiday"] = test["is_holiday"].fillna(0).astype(int)

    return train, test

## Calendar Features

In [6]:
def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["dayofweek"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["day"] = df["date"].dt.day
    return df

## Adding lag features

In [7]:
def add_train_time_features(train: pd.DataFrame) -> pd.DataFrame:
    train = train.copy()
    train = train.sort_values(["store_nbr", "family", "date"])

    grp = train.groupby(["store_nbr", "family"])["sales"]

    train["lag_1"] = grp.shift(1)
    train["lag_7"] = grp.shift(7)

    # remove rows that cannot form the lag / rolling features
    train = train.dropna(subset=["lag_1", "lag_7"])

    return train

## One hot encoding categorical features

In [8]:
def encode_features(train: pd.DataFrame, test: pd.DataFrame):
    train = train.copy()
    test = test.copy()

    categorical_cols = ["family", "city", "state", 'type']

    # add placeholder sales to test so concatenation is easy
    test["sales"] = np.nan
    test["lag_1"] = np.nan
    test["lag_7"] = np.nan

    combined = pd.concat([train, test], axis=0, ignore_index=True)
    combined = pd.get_dummies(combined, columns=categorical_cols)

    # LightGBM dislikes special characters in feature names
    combined.columns = combined.columns.str.replace(r"[^A-Za-z0-9_]", "_", regex=True)

    train_encoded = combined[combined["sales"].notna()].copy()
    test_encoded = combined[combined["sales"].isna()].copy()

    return train_encoded, test_encoded

## Run baseline model

In [20]:
import lightgbm as lgb
from sklearn.metrics import mean_squared_log_error


def run_baseline_model(train: pd.DataFrame):
    train = train.copy()

    train_data = train[train["date"] < "2017-01-01"].copy()
    val_data = train[train["date"] >= "2017-01-01"].copy()

    drop_cols = ["sales", "date", "id"]
    X_train = train_data.drop(columns=drop_cols, errors="ignore")
    X_val = val_data.drop(columns=drop_cols, errors="ignore")

    y_train_raw = train_data["sales"]
    y_val_raw = val_data["sales"]

    y_train_log = np.log1p(y_train_raw)
    y_val_log = np.log1p(y_val_raw)

    # numeric safety
    X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)
    X_train = X_train.astype(float)
    X_val = X_val.astype(float)

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=1500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    )

    model.fit(
        X_train,
        y_train_log,
        eval_set=[(X_val, y_val_log)],
        eval_metric="l2",
        callbacks=[
            lgb.log_evaluation(period=50),
            lgb.early_stopping(stopping_rounds=500)
        ]
    )

    y_pred_log = model.predict(X_val)
    y_pred = np.expm1(y_pred_log)
    y_pred = np.clip(y_pred, 0, None)

    rmsle = np.sqrt(mean_squared_log_error(y_val_raw, y_pred))
    print("RMSLE:", rmsle)

    return model, rmsle

## Full pipeline

In [21]:
holiday, oil, train, test, stores, transactions = load_dataset()

oil = preprocess_oil(oil)
holiday_feat = preprocess_holiday_baseline(holiday)

train, test = merge_base_features(train, test, oil, stores, holiday_feat[['date', 'is_holiday']])

train = add_calendar_features(train)
test = add_calendar_features(test)

train = add_train_time_features(train)

train_encoded, test_encoded = encode_features(train, test)

model, rmsle = run_baseline_model(train_encoded)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022325 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1243
[LightGBM] [Info] Number of data points in the train set: 2630232, number of used features: 87
[LightGBM] [Info] Start training from score 2.833966
Training until validation scores don't improve for 500 rounds
[50]	valid_0's l2: 0.320399
[100]	valid_0's l2: 0.248578
[150]	valid_0's l2: 0.243132
[200]	valid_0's l2: 0.238754
[250]	valid_0's l2: 0.23662
[300]	valid_0's l2: 0.235647
[350]	valid_0's l2: 0.234704
[400]	valid_0's l2: 0.233405
[450]	valid_0's l2: 0.232278
[500]	valid_0's l2: 0.231377
[550]	valid_0's l2: 0.230353
[600]	valid_0's l2: 0.229806
[650]	valid_0's l2: 0.22911
[700]	valid_0's l2: 0.228488
[750]	valid_0's l2: 0.227888
[800]	valid_0's l2: 0.227315
[850]	valid_0's l2: 0.226885
[900]	valid_0's l2: 0.226492
[950]	valid

## Data Cleaning

In [11]:
# holiday, oil, train, test, stores, transaction = load_dataset()

# dataframes = {
#     "train": train,
#     "test": test,
#     "oil": oil,
#     "holidays": holiday,
#     "stores": stores,
#     "transactions": transaction
# }

# for name, df in dataframes.items():
#     print(f'\n{name.upper()}')
#     print(df.isnull().sum())

# # Oil prices are not reported on weekends, therefore has a lot of nulls
# # Therefore we fill with the last known oil price
# def data_cleaning_oil(oil):
#     oil['dcoilwtico'] = oil['dcoilwtico'].ffill()
#     return oil

# for name, df in dataframes.items():
#     print(f'\n{name.upper()}')
#     print(df.isnull().sum())

## Merging datasets

In [12]:
# def merging_dataset(oil, train, test, stores):
#     # Merge train and test to oil dataframe
#     train = train.merge(oil, on='date', how='left')
#     test = test.merge(oil, on='date', how='left')

#     # Merge train data and stores dataframe
#     train = train.merge(stores, on='store_nbr', how='left')    
#     return train

In [13]:
# def simple_holiday_preprocessing(holiday, train, test):
#     holiday['is_holiday'] = holiday['type'] != "Work Day"
#     holiday['is_holiday'] = holiday['is_holiday'].astype(int)
#     train = train.merge(holiday[['date', 'is_holiday']], on='date', how='left')
#     test = test.merge(holiday[['date', 'is_holiday']], on="date", how="left")
#     train["is_holiday"] = train["is_holiday"].fillna(0)
#     test["is_holiday"] = test["is_holiday"].fillna(0)
#     return holiday, train, test


## Data Preprocessing

In [14]:
# def data_preprocess_train_dataset(train, categorical):
#     # Sort values for time lag features
#     train = train.sort_values(['store_nbr', 'family', 'date'])

#     # Creating lag features to learn from past sales
#     train['lag_1'] = train.groupby(['store_nbr', 'family'])['sales'].shift(1)
#     train['lag_7'] = train.groupby(['store_nbr', 'family'])['sales'].shift(7)
#     train = train.dropna(subset=['lag_1', 'lag_7'])

#     # Feature to breakdown date
#     train['dayofweek'] = train['date'].dt.dayofweek
#     train['month'] = train['date'].dt.month
#     train['year'] = train['date'].dt.year

#     # Encode categorical to numerical
#     categorical = ['family', 'city', 'state', 'type']
#     train = pd.get_dummies(train, columns=categorical)

#     # remove invalid characters in column names
#     train.columns = train.columns.str.replace(r"[^A-Za-z0-9_]", "_", regex=True)

#     return train


## Model Prediction (Baseline)

In [15]:
# import numpy as np
# import lightgbm as lgb
# from sklearn.metrics import mean_squared_log_error

# def run_prediction(train):
#     train_data = train[train.date < "2017-01-01"].copy()
#     val_data = train[train.date >= "2017-01-01"].copy()

#     X_train = train_data.drop(columns=["sales", "date"])
#     X_val = val_data.drop(columns=["sales", "date"])

#     y_train_raw = train_data["sales"]
#     y_val_raw = val_data["sales"]

#     y_train_log = np.log1p(y_train_raw)
#     y_val_log = np.log1p(y_val_raw)

#     X_train = X_train.astype(float)
#     X_val = X_val.astype(float)

#     model = lgb.LGBMRegressor(
#         objective="regression",
#         n_estimators=1000,
#         learning_rate=0.05,
#         num_leaves=31,
#         random_state=42
#     )

#     model.fit(
#         X_train,
#         y_train_log,
#         eval_set=[(X_val, y_val_log)],
#         eval_metric="l2",
#         callbacks=[
#             lgb.log_evaluation(period=50),
#             lgb.early_stopping(stopping_rounds=100)
#         ]
#     )

#     y_pred_log = model.predict(X_val)
#     y_pred = np.expm1(y_pred_log)
#     y_pred = np.clip(y_pred, 0, None)

#     rmsle = np.sqrt(mean_squared_log_error(y_val_raw, y_pred))
#     print("RMSLE:", rmsle)
#     return rmsle

## Run 1st pipeline

In [16]:
# holiday, oil, train, test, stores, transaction = load_dataset()
# oil = data_cleaning_oil(oil)
# train = merging_dataset(oil, train, test, stores)
# holiday, train, test = simple_holiday_preprocessing(holiday, train, test)

# categorical = ['family', 'city', 'state', 'type']
# train = data_preprocess_train_dataset(train, categorical)
# rmsle = run_prediction(train)

## Feature Engineering

In [17]:
# holiday, oil, train, test, stores, transaction = load_dataset()
# oil = data_cleaning_oil(oil)
# train

In [18]:
# holiday, oil, train, test, stores, transaction = load_dataset()
# oil = data_cleaning_oil(oil)

# # Creating new features from the holiday dataset
# train = merging_dataset(oil, train, test, stores)

# train = holiday[holiday['transferred'] == False]
# holiday['is_bridge'] = holiday['type'] == 'Bridge'
# holiday['is_event'] = holiday['type'] == 'Event'
# holiday['is_workday'] = holiday['type'] == 'Work Day'

# train = train.merge(holiday, on='date', how='left')

# train['is_holiday'] = (
#     (train['locale'] == 'National') | 
#     (train['locale'] == 'Regional' & train['state'] == train['locale_name']) |
#     (train['locale'] == 'Local' & train['city'] == train['locale_name'])
# )

# # Can try this later
# # is_national_holiday
# # is_regional_holiday
# # is_local_holiday
# # is_event
# # is_bridge

In [19]:
# subset = train_df[
#     (train_df.store_nbr == 1) &
#     (train_df.family == "DAIRY")
# ].shape

# print(subset)